In [1]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2 as cv

#Impoting the image
img = cv.imread('heroine.jpg')

#Changing the image to gray scale
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

#Innitialize SIFT detector
sift = cv.SIFT_create()

#Detecting key points of the image
kp = sift.detect(gray, None)

#Drawing the keypoints on the image
img = cv.drawKeypoints(gray, kp, img)

#DIsplay image with keypoints
cv.imshow('SIFT Keypoints', img)
cv.waitKey(0)
cv.imwrite('sift_keypoints.jpg', img)


True

In [3]:
import numpy as np

#Create a mask initaialise with zero
mask = np.zeros(gray.shape, dtype=np.uint8)

#Define ROI coordinates
x_start,y_start = 290,170
x_end,y_end = 456,368

#Set the ROI area in the mask to 255
mask[y_start:y_end, x_start:x_end] = 255

#Initialize SIFT detector
sift = cv.SIFT_create()

#Detect keypoints using mask
kp = sift.detect(gray,mask)

# Draw keypoints on the image
img_with_kp = cv.drawKeypoints(img, kp, None)

# Highlight the ROI with a rectangle (in red for visibility)
cv.rectangle(img_with_kp, (x_start, y_start), (x_end, y_end), (0, 0, 255), 2) 

# Display the image with keypoints and ROI
cv.imshow('SIFT Keypoints with ROI', img_with_kp)
cv.waitKey(0)

-1

######
 it computes 128-dimensional descriptor vectors for the SIFT keypoints detected earlier, encoding local image gradient patterns around each keypoint in a rotation- and scale-invariant way.

In [5]:
kp, des = sift.detectAndCompute(gray, mask)
if des is not None:
    print(f"Found {len(kp)} keypoints with descriptors shape: {des.shape}")
else:
    print("No keypoints or descriptors found—try enlarging ROI or reducing mask constraints.")

No keypoints or descriptors found—try enlarging ROI or reducing mask constraints.


In [ ]:
import cv2
import numpy as np
from PIL import Image
import io

def compute_ela(image_path, quality=90):
    """Compute Error Level Analysis."""
    original = Image.open(image_path)
    original_save = io.BytesIO()
    original.save(original_save, format='JPEG', quality=quality, optimize=True)
    resaved = Image.open(io.BytesIO(original_save.getvalue()))
    
    # Convert to arrays
    orig_array = np.array(original).astype(np.float32)
    resaved_array = np.array(resaved).astype(np.float32)
    
    # Compute ELA (absolute difference)
    ela = np.abs(orig_array - resaved_array)
    # Scale for visibility
    max_val = ela.max()
    if max_val > 0:
        ela = (ela / max_val) * 255
    return np.uint8(ela)

# Example usage (replace 'your_image.jpg' with path)
image_path = 'your_image.jpg'  # e.g., 'heroine.jpg'
ela = compute_ela(image_path)

# Save ELA
cv2.imwrite('ela_output.png', cv2.cvtColor(ela, cv2.COLOR_RGB2BGR))

# Frequency analysis: FFT magnitude spectrum
img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
f = np.fft.fft2(gray)
fshift = np.fft.fftshift(f)
magnitude_spectrum = 20 * np.log(np.abs(fshift))
cv2.imwrite('frequency_spectrum.png', magnitude_spectrum)

print("ELA saved as 'ela_output.png' – uniform blue/green in real images.")
print("Frequency spectrum saved as 'frequency_spectrum.png' – check high-freq noise.")
